In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score
from sklearn.model_selection import StratifiedKFold
from pyod.models.deep_svdd import DeepSVDD
import sys
import scrapbook as sb

sys.path.append('..')
from utils import reshape_to_numpy, interpolate_missing_values, denoise_data, compute_spectrograms, time_avg_pooling

In [2]:
seed = 1
sampling_rate = 10
nperseg=64
noverlap=32
accel_cutoff = 0.4
accel_order = 30
gyro_cutoff = 0.6
gyro_order = 20
batch_size = 32
epochs = 100
hidden_neurons = [128, 64, 32]
n_splits = 5

In [3]:
# Parameters
seed = 20


In [4]:
rng = np.random.RandomState(seed)

In [5]:
data = pd.read_parquet("../data/GBG500.parquet")
data

,ride_id,time_index,ax,ay,az,rx,ry,rz
0,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,0,-2.512796,-9.385012,-1.053078,-0.009155,0.009155,-0.201416
1,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,100,-2.491268,-9.385012,-1.079390,-0.036621,0.027465,-0.155639
2,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,200,-2.534324,-9.382022,-1.030952,0.036621,0.027465,-0.073242
3,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,300,-2.488278,-9.383218,-1.091948,-0.045776,0.036621,-0.183105
4,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,400,-2.483494,-9.382022,-1.100320,-0.045776,0.027465,-0.192260
...,...,...,...,...,...,...,...,...
1529542,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241300,0.459862,-9.140430,-3.318900,1.556396,-0.091552,0.274658
1529543,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241400,0.455676,-9.140430,-3.321292,1.583861,-0.119018,0.274658
1529544,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241500,0.459862,-9.139832,-3.317704,1.583861,-0.109863,0.274658
1529545,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241600,0.456274,-9.142224,-3.321292,1.574707,-0.137329,0.283813


In [6]:
labels = pd.read_csv("../data/GBG500_labels.csv")
labels.columns = labels.columns.str.lower()
ride_order_df = pd.DataFrame({"ride_id": data["ride_id"].unique()})
labels_sorted = ride_order_df.merge(
    labels,
    on="ride_id",
    how="left"
)
labels_sorted

,ride_id,label
0,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,Safe
1,00d223cb7aecc9c0cc5871e32a6c45027a068eba07c572...,Reckless
2,02288a4aeca044203394e982e76b87818021dea2b34df9...,Safe
3,0236bdcf13d473ea24d97f5eaeea459f257dffac005694...,Bad weather
4,0238e5dd85143f1b6c59b200bc32f14361b8fc84c51a87...,Safe
...,...,...
495,fe5d7f84d692bbccad8bd566a3ad5c3108fa9f90acd671...,Safe
496,fe67169632306d4668b2affedef510df2cb43e2fb41220...,Safe
497,fee44667fdc8ab4995c702fc3bd36178de0b1ca2c64687...,Safe
498,ff79b2e945b93c2a5efc3a06365267b3a160e7849ba57d...,Safe


In [7]:
data_np = reshape_to_numpy(
    data,
    features = ["ax", "ay", "az", "rx", "ry", "rz"],
    max_timestamps = 4800
)

data_np_clean = interpolate_missing_values(
    data_np,
    method='linear',
    limit=None
)

data_np_clean = denoise_data(
    data=data_np_clean,
    accel_indices=[0, 1, 2],
    gyro_indices=[3, 4, 5],
    accel_cutoff=accel_cutoff,
    accel_order=accel_order,
    gyro_cutoff=gyro_cutoff,
    gyro_order=gyro_order,
)

In [8]:
# Compute spectrograms for all rides
spectrograms_array = compute_spectrograms(data_np_clean, sampling_rate, nperseg, noverlap)
spectrograms_array.shape

(500, 149, 6, 33)

In [9]:
# Aggregate spectrograms using time-averaged pooling
X_feat = time_avg_pooling(spectrograms_array)
X_feat.shape

(500, 198)

In [10]:
# 5-fold stratified CV with Deep SVDD
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=rng.randint(1000))
y_true = (labels_sorted['label'] == 'Reckless').astype(int).values
anomaly_scores = np.full(len(y_true), np.nan)

for fold, (train_idx, test_idx) in enumerate(skf.split(X_feat, y_true)):
    sc = StandardScaler()
    X_train = sc.fit_transform(X_feat[train_idx])
    X_test = sc.transform(X_feat[test_idx])

    np.random.seed(rng.randint(1000))
    model = DeepSVDD(
        n_features=X_train.shape[1],
        hidden_neurons=hidden_neurons,
        epochs=epochs,
        batch_size=batch_size,
        random_state=rng.randint(1000),
    )
    model.fit(X_train)
    anomaly_scores[test_idx] = model.decision_function(X_test)
    print(f"Fold {fold+1}/{n_splits} done")

Epoch 1/100, Loss: 1.4699185080826283
Epoch 2/100, Loss: 1.5999921187758446
Epoch 3/100, Loss: 1.5272769294679165
Epoch 4/100, Loss: 1.3943945802748203
Epoch 5/100, Loss: 1.5146292932331562
Epoch 6/100, Loss: 1.3412707149982452
Epoch 7/100, Loss: 1.3409657217562199
Epoch 8/100, Loss: 1.4419353753328323
Epoch 9/100, Loss: 1.3912780955433846
Epoch 10/100, Loss: 1.3903102204203606
Epoch 11/100, Loss: 1.409800447523594
Epoch 12/100, Loss: 1.4591867625713348
Epoch 13/100, Loss: 1.3313940428197384
Epoch 14/100, Loss: 1.421314887702465
Epoch 15/100, Loss: 1.4248622320592403
Epoch 16/100, Loss: 1.407438401132822
Epoch 17/100, Loss: 1.505916003137827


Epoch 18/100, Loss: 1.4823353849351406
Epoch 19/100, Loss: 1.3996905833482742
Epoch 20/100, Loss: 1.3757333867251873
Epoch 21/100, Loss: 1.4441151171922684
Epoch 22/100, Loss: 1.3301136568188667
Epoch 23/100, Loss: 1.4479887522757053
Epoch 24/100, Loss: 1.3440689332783222
Epoch 25/100, Loss: 1.372135415673256
Epoch 26/100, Loss: 1.4867753237485886
Epoch 27/100, Loss: 1.3626966699957848
Epoch 28/100, Loss: 1.426225669682026
Epoch 29/100, Loss: 1.3897574543952942
Epoch 30/100, Loss: 1.45130155980587
Epoch 31/100, Loss: 1.3781579658389091
Epoch 32/100, Loss: 1.5370297692716122
Epoch 33/100, Loss: 1.3341297954320908
Epoch 34/100, Loss: 1.3275795429944992


Epoch 35/100, Loss: 1.4130472615361214
Epoch 36/100, Loss: 1.4520630091428757
Epoch 37/100, Loss: 1.4208791330456734
Epoch 38/100, Loss: 1.4376038648188114
Epoch 39/100, Loss: 1.5280981920659542
Epoch 40/100, Loss: 1.4323061369359493
Epoch 41/100, Loss: 1.695399422198534
Epoch 42/100, Loss: 1.4542937204241753
Epoch 43/100, Loss: 1.3431299030780792
Epoch 44/100, Loss: 1.447010077536106
Epoch 45/100, Loss: 1.431582197546959
Epoch 46/100, Loss: 1.5685145556926727
Epoch 47/100, Loss: 1.4067529141902924
Epoch 48/100, Loss: 1.3419338688254356
Epoch 49/100, Loss: 1.4413669854402542
Epoch 50/100, Loss: 1.4590611942112446
Epoch 51/100, Loss: 1.377710223197937
Epoch 52/100, Loss: 1.5052483826875687


Epoch 53/100, Loss: 1.5020154230296612
Epoch 54/100, Loss: 1.3264003843069077
Epoch 55/100, Loss: 1.380019187927246
Epoch 56/100, Loss: 1.5464601330459118
Epoch 57/100, Loss: 1.4160213470458984
Epoch 58/100, Loss: 1.4607782065868378
Epoch 59/100, Loss: 1.4508064836263657
Epoch 60/100, Loss: 1.5267806760966778
Epoch 61/100, Loss: 1.4971781484782696
Epoch 62/100, Loss: 1.3413494378328323
Epoch 63/100, Loss: 1.386999063193798
Epoch 64/100, Loss: 1.3978578373789787
Epoch 65/100, Loss: 1.438699595630169
Epoch 66/100, Loss: 1.4171634837985039
Epoch 67/100, Loss: 1.548247143626213
Epoch 68/100, Loss: 1.4807527773082256
Epoch 69/100, Loss: 1.4495685510337353
Epoch 70/100, Loss: 1.3516088612377644


Epoch 71/100, Loss: 1.3583399020135403
Epoch 72/100, Loss: 1.3057403266429901
Epoch 73/100, Loss: 1.451989971101284
Epoch 74/100, Loss: 1.3305228166282177
Epoch 75/100, Loss: 1.430062223225832
Epoch 76/100, Loss: 1.3809420205652714
Epoch 77/100, Loss: 1.3935480415821075
Epoch 78/100, Loss: 1.4515390507876873
Epoch 79/100, Loss: 1.6064291298389435
Epoch 80/100, Loss: 1.458208929747343
Epoch 81/100, Loss: 1.5172584280371666
Epoch 82/100, Loss: 1.360037125647068
Epoch 83/100, Loss: 1.3460934795439243
Epoch 84/100, Loss: 1.6532228887081146
Epoch 85/100, Loss: 1.4646172150969505
Epoch 86/100, Loss: 1.3121405392885208
Epoch 87/100, Loss: 1.5347231775522232


...

Epoch 44/100, Loss: 2.677545666694641
Epoch 45/100, Loss: 2.808522216975689
Epoch 46/100, Loss: 2.8353133127093315
Epoch 47/100, Loss: 2.6103453636169434
Epoch 48/100, Loss: 2.8684555888175964
Epoch 49/100, Loss: 2.6270106807351112
Epoch 50/100, Loss: 2.865021660923958
Epoch 51/100, Loss: 2.521541118621826
Epoch 52/100, Loss: 2.6970072090625763
Epoch 53/100, Loss: 2.496687598526478
Epoch 54/100, Loss: 2.788977660238743
Epoch 55/100, Loss: 2.557224787771702


Epoch 56/100, Loss: 2.8042506724596024
Epoch 57/100, Loss: 2.6728782951831818
Epoch 58/100, Loss: 2.742834448814392
Epoch 59/100, Loss: 3.0189418122172356
Epoch 60/100, Loss: 2.7071026414632797
Epoch 61/100, Loss: 2.6215164959430695
Epoch 62/100, Loss: 2.689526744186878
Epoch 63/100, Loss: 2.7180833742022514
Epoch 64/100, Loss: 2.834900550544262
Epoch 65/100, Loss: 2.708151489496231
Epoch 66/100, Loss: 2.7500180676579475


Epoch 67/100, Loss: 2.628180406987667
Epoch 68/100, Loss: 2.7751971632242203
Epoch 69/100, Loss: 2.634190984070301
Epoch 70/100, Loss: 2.929655075073242
Epoch 71/100, Loss: 2.8231044188141823
Epoch 72/100, Loss: 2.7021605148911476
Epoch 73/100, Loss: 2.7054394632577896
Epoch 74/100, Loss: 2.64185394346714
Epoch 75/100, Loss: 2.5804311335086823
Epoch 76/100, Loss: 2.8842284455895424
Epoch 77/100, Loss: 3.463115468621254


Epoch 78/100, Loss: 2.7161010950803757
Epoch 79/100, Loss: 2.631595440208912
Epoch 80/100, Loss: 2.7095434367656708
Epoch 81/100, Loss: 2.560461178421974
Epoch 82/100, Loss: 2.825016550719738
Epoch 83/100, Loss: 2.8333645164966583
Epoch 84/100, Loss: 3.0410724133253098
Epoch 85/100, Loss: 2.8990058228373528
Epoch 86/100, Loss: 2.710686206817627
Epoch 87/100, Loss: 2.6475762128829956
Epoch 88/100, Loss: 2.7618933990597725
Epoch 89/100, Loss: 2.8210798650979996
Epoch 90/100, Loss: 2.690017990767956
Epoch 91/100, Loss: 2.8007419034838676
Epoch 92/100, Loss: 2.718482129275799


Epoch 93/100, Loss: 2.762413941323757
Epoch 94/100, Loss: 2.9964373484253883
Epoch 95/100, Loss: 2.7936930507421494
Epoch 96/100, Loss: 2.7961505651474
Epoch 97/100, Loss: 2.857855163514614
Epoch 98/100, Loss: 2.658822387456894
Epoch 99/100, Loss: 2.7964605391025543
Epoch 100/100, Loss: 2.6911889612674713
Fold 5/5 done


In [11]:
ap = average_precision_score(y_true, anomaly_scores)
print(f"Deep SVDD AP (5-fold CV) = {ap:.4f}")
sb.glue("GBG500_ap_spectral_deep_svdd", float(ap))

Deep SVDD AP (5-fold CV) = 0.4511
